In [ ]:
# WEEK 4 — LLM RELIABILITY + NETIXSOL CAPSTONE
# Run this notebook from the existing web3geeks-ai-bootcamp project.
# Gemini is used because the mentor approved it as the free alternative to Anthropic.

import os
import re
import json
import time
import csv
import requests
import numpy as np
import pandas as pd
import joblib
import chromadb

from pathlib import Path
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from pydantic import BaseModel, Field, create_model
from enum import Enum
from google import genai
from google.genai import types

# Locate repository root whether Jupyter starts in repo root or week4/
current_dir = Path.cwd()
if (current_dir / "week3").exists():
    project_dir = current_dir
elif (current_dir.parent / "week3").exists():
    project_dir = current_dir.parent
else:
    raise FileNotFoundError(
        "Cannot locate the repository root. Open this notebook from "
        "F:\\web3geeks-ai-bootcamp or its week4 folder."
    )

week4_dir = project_dir / "week4"
prompts_dir = week4_dir / "prompts"
results_dir = week4_dir / "results"
prompts_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

# Load key from week4/.env. Never paste the key into this notebook.
load_dotenv(week4_dir / ".env", override=True)
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

gemini_client = None
if GEMINI_API_KEY:
    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    print("Gemini client initialized successfully.")
else:
    print("Gemini key not found yet. Non-API cells can still run.")
    print("Add GEMINI_API_KEY=your_key to week4/.env before API/RAG cells.")

GENERATION_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "gemini-embedding-001")

def require_gemini():
    if gemini_client is None:
        raise RuntimeError(
            "Gemini API key missing. Add GEMINI_API_KEY to week4/.env, "
            "then rerun Cell 0."
        )

print("Project root:", project_dir)
print("Week 4 folder:", week4_dir)


def retry_gemini_call(call, label="Gemini API", attempts=3):
    """Retry transient Gemini failures up to three attempts (2s, then 4s)."""
    waits = (2, 4)
    for attempt in range(1, attempts + 1):
        try:
            return call()
        except Exception as exc:
            message = str(exc).upper()
            transient = any(code in message for code in (
                "503", "UNAVAILABLE", "429", "RESOURCE_EXHAUSTED",
                "500", "502", "504", "INTERNAL", "TIMEOUT"
            ))
            if not transient or attempt == attempts:
                raise
            delay = waits[attempt - 1]
            print(f"{label} temporarily unavailable; retrying in {delay}s ({attempt + 1}/{attempts})")
            time.sleep(delay)


In [2]:
# First Gemini API request (replaces the Anthropic request because the mentor approved Gemini)
require_gemini()

response = gemini_client.models.generate_content(
    model=GENERATION_MODEL,
    contents="Explain artificial intelligence in simple words with one real-life example."
)

print("ANSWER:")
print(response.text)
print("\nUSAGE METADATA:")
print(response.usage_metadata)


TypeError: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for the `Authorization` header to be explicitly omitted"

In [3]:
# Temperature test: Gemini supports temperature=0 in generation config.
# This is the equivalent reproducibility experiment for the approved Gemini model.

require_gemini()

temperature_response = gemini_client.models.generate_content(
    model=GENERATION_MODEL,
    contents="Classify the sentiment of this sentence as Positive, Negative, or Neutral: I love this service.",
    config=types.GenerateContentConfig(
        temperature=0,
        max_output_tokens=100
    )
)

print("Temperature 0 response:")
print(temperature_response.text)
print("\nExplanation: temperature controls randomness. A value of 0 makes output more deterministic,")
print("although identical output is not guaranteed in every model/version.")


ERROR TYPE: TypeError
ERROR MESSAGE: Messages.create() got an unexpected keyword argument 'temperature'


In [5]:
from pathlib import Path

print("Current working directory:", Path.cwd())

Current working directory: f:\web3geeks-ai-bootcamp\week4


In [6]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import time

# 1. Automatically locate the project folder
current_dir = Path.cwd()

if (current_dir / "week3").exists():
    project_dir = current_dir
elif (current_dir.parent / "week3").exists():
    project_dir = current_dir.parent
else:
    raise FileNotFoundError("Cannot locate the week3 folder.")

# 2. Define file paths
train_path = project_dir / "week3/challenge/data/train.csv"
test_path = project_dir / "week3/challenge/data/test.csv"
model_path = project_dir / "week3/challenge/model.joblib"

# 3. Load Banking77 datasets
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

# 4. Select the exact 200 evaluation questions
sample_200 = test.sample(
    200,
    random_state=42
).reset_index(drop=True)

# 5. Verify the first question
assert sample_200.iloc[0]["text"] == "How do I link this new card?"
assert sample_200.iloc[0]["category"] == "card_linking"

# 6. Load the existing challenge model
model = joblib.load(model_path)

# 7. Verify all 77 intent categories
assert len(model.classes_) == 77
assert set(model.classes_) == set(train["category"].unique())

# 8. Generate predictions and probabilities
start_time = time.perf_counter()

probabilities = model.predict_proba(sample_200["text"])

elapsed_time = time.perf_counter() - start_time

# 9. Get predicted intents and confidence scores
predicted_indices = np.argmax(probabilities, axis=1)
predicted_labels = model.classes_[predicted_indices]
confidence_scores = np.max(probabilities, axis=1)

# 10. Create evaluation results
results = sample_200.copy()

results["predicted_intent"] = predicted_labels
results["confidence"] = confidence_scores
results["correct"] = (
    results["category"] == results["predicted_intent"]
)

results["needs_human"] = results["confidence"] < 0.40

# 11. Calculate performance
accuracy = results["correct"].mean() * 100
human_count = results["needs_human"].sum()
seconds_per_message = elapsed_time / len(sample_200)

# 12. Save results
output_path = project_dir / "week4/challenge_model_results.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

results.to_csv(output_path, index=False)

# 13. Display results
print("BANKING77 CHALLENGE MODEL RESULTS")
print("---------------------------------")
print("Total questions:", len(results))
print(f"Accuracy: {accuracy:.2f}%")
print("Sent to human:", human_count)
print(f"Average seconds per message: {seconds_per_message:.6f}")
print("Results saved to:", output_path)

display(results.head())

BANKING77 CHALLENGE MODEL RESULTS
---------------------------------
Total questions: 200
Accuracy: 83.50%
Sent to human: 102
Average seconds per message: 0.000225
Results saved to: f:\web3geeks-ai-bootcamp\week4\challenge_model_results.csv


,text,category,predicted_intent,confidence,correct,needs_human
0,How do I link this new card?,card_linking,card_linking,0.839596,True,False
1,How do I retrieve my card from the machine?,card_swallowed,card_not_working,0.056942,False,True
2,I want to know where the funds come from.,verify_source_of_funds,verify_source_of_funds,0.724310,True,False
3,"I just activated auto top-up, but it is not le...",automatic_top_up,automatic_top_up,0.166140,True,True
4,Why did I have to pay extra because I paid wit...,card_payment_fee_charged,card_payment_fee_charged,0.429267,True,False


In [7]:
# 1. Get all 77 intent categories
intent_names = sorted(train["category"].unique())

print("Total intent categories:", len(intent_names))

# 2. Display all intent names
print("\nALL 77 INTENTS:")
for number, intent in enumerate(intent_names, start=1):
    print(f"{number}. {intent}")

# 3. Select one training example for each intent
few_shot_examples = (
    train.groupby("category")
    .sample(n=1, random_state=42)
    .sort_values("category")
    .reset_index(drop=True)
)

# 4. Verify we have one example per intent
assert len(few_shot_examples) == 77
assert few_shot_examples["category"].nunique() == 77

print("\nFEW-SHOT EXAMPLES")
print("-----------------")
print("Total examples:", len(few_shot_examples))

display(few_shot_examples.head(10))

# 5. Save examples for later API experiments
examples_path = project_dir / "week4/few_shot_examples.csv"

few_shot_examples.to_csv(examples_path, index=False)

print("\nExamples saved to:", examples_path)

Total intent categories: 77

ALL 77 INTENTS:
1. Refund_not_showing_up
2. activate_my_card
3. age_limit
4. apple_pay_or_google_pay
5. atm_support
6. automatic_top_up
7. balance_not_updated_after_bank_transfer
8. balance_not_updated_after_cheque_or_cash_deposit
9. beneficiary_not_allowed
10. cancel_transfer
11. card_about_to_expire
12. card_acceptance
13. card_arrival
14. card_delivery_estimate
15. card_linking
16. card_not_working
17. card_payment_fee_charged
18. card_payment_not_recognised
19. card_payment_wrong_exchange_rate
20. card_swallowed
21. cash_withdrawal_charge
22. cash_withdrawal_not_recognised
23. change_pin
24. compromised_card
25. contactless_not_working
26. country_support
27. declined_card_payment
28. declined_cash_withdrawal
29. declined_transfer
30. direct_debit_payment_not_recognised
31. disposable_card_limits
32. edit_personal_details
33. exchange_charge
34. exchange_rate
35. exchange_via_app
36. extra_charge_on_statement
37. failed_transfer
38. fiat_currency_suppor

,text,category
0,I made a big purchase that I regretted and can...,Refund_not_showing_up
1,i cant seem to activate card,activate_my_card
2,My daughter needs to open account.,age_limit
3,Are top-ups allowed using Apple Pay?,apple_pay_or_google_pay
4,I need to know where I can withdraw money.,atm_support
5,Where do I access the auto top-up function?,automatic_top_up
6,I'm wondering how long it takes for my money t...,balance_not_updated_after_bank_transfer
7,I made a cash deposit a few days ago and it's ...,balance_not_updated_after_cheque_or_cash_deposit
8,Why can't I transfer to an account?,beneficiary_not_allowed
9,is it possible to cancel transactions?,cancel_transfer



Examples saved to: f:\web3geeks-ai-bootcamp\week4\few_shot_examples.csv


In [8]:
# Zero-shot system prompt
zero_shot_system = f"""
You are a customer support intent classification assistant.

Your task is to classify each customer message into exactly ONE
of the following 77 intent categories:

{", ".join(intent_names)}

Rules:
1. Choose only one category from the list.
2. Return the exact category name.
3. Do not explain your answer.
4. Do not add punctuation or extra text.
5. If uncertain, choose the most likely category.
"""

print("Zero-shot system prompt prepared.")
print("Available intent categories:", len(intent_names))
print("\nPrompt preview:\n")
print(zero_shot_system[:500] + "...")

Zero-shot system prompt prepared.
Available intent categories: 77

Prompt preview:


You are a customer support intent classification assistant.

Your task is to classify each customer message into exactly ONE
of the following 77 intent categories:

Refund_not_showing_up, activate_my_card, age_limit, apple_pay_or_google_pay, atm_support, automatic_top_up, balance_not_updated_after_bank_transfer, balance_not_updated_after_cheque_or_cash_deposit, beneficiary_not_allowed, cancel_transfer, card_about_to_expire, card_acceptance, card_arrival, card_delivery_estimate, card_linking, ca...


In [9]:
# Few-shot system prompt
few_shot_system = f"""
You are a customer support intent classification assistant.

Classify each customer message into exactly ONE of these 77 categories:

{", ".join(intent_names)}

Here are examples of correctly classified messages:

"""

# Add one example for every intent
for _, row in few_shot_examples.iterrows():
    few_shot_system += (
        f"\nCustomer message: {row['text']}"
        f"\nIntent: {row['category']}\n"
    )

few_shot_system += """
Rules:
1. Choose exactly one category.
2. Use the exact category name from the list.
3. Do not explain your answer.
4. Do not add punctuation or extra text.
5. If uncertain, choose the most likely category.
"""

print("Few-shot system prompt prepared.")
print("Intent categories:", len(intent_names))
print("Examples included:", len(few_shot_examples))
print("Prompt length:", len(few_shot_system), "characters")

Few-shot system prompt prepared.
Intent categories: 77
Examples included: 77
Prompt length: 10306 characters


In [10]:
from enum import Enum
from pydantic import BaseModel, Field, create_model

# 1. Create an Enum containing only the 77 valid intents
IntentEnum = Enum(
    "IntentEnum",
    {f"INTENT_{i}": name for i, name in enumerate(intent_names)}
)

# 2. Define the required structured response
ClassificationOutput = create_model(
    "ClassificationOutput",
    intent=(IntentEnum, ...),
    confidence=(float, Field(ge=0, le=1)),
    needs_human=(bool, ...)
)

# 3. Test the model locally
test_output = ClassificationOutput(
    intent=IntentEnum("card_linking"),
    confidence=0.95,
    needs_human=False
)

print("Structured output model created successfully.")
print(test_output.model_dump())
print("\nValid intent choices:", len(IntentEnum))

Structured output model created successfully.
{'intent': <IntentEnum.INTENT_14: 'card_linking'>, 'confidence': 0.95, 'needs_human': False}

Valid intent choices: 77


In [11]:
import json

# Folder for experiment results
results_dir = project_dir / "week4/results"
results_dir.mkdir(parents=True, exist_ok=True)

# CSV files for each experiment
zero_shot_file = results_dir / "zero_shot_results.csv"
few_shot_file = results_dir / "few_shot_results.csv"
structured_file = results_dir / "structured_results.csv"

def save_result(result, file_path):
    """Save one result immediately to CSV."""
    new_row = pd.DataFrame([result])

    if file_path.exists():
        new_row.to_csv(
            file_path,
            mode="a",
            header=False,
            index=False
        )
    else:
        new_row.to_csv(file_path, index=False)

print("Result-saving system prepared.")
print("Results folder:", results_dir)

Result-saving system prepared.
Results folder: f:\web3geeks-ai-bootcamp\week4\results


## Part B — Netixsol AI Client Intake & Project Scoping Agent

Mentor approval: public Netixsol website pages may be used as knowledge sources. Gemini is the approved free alternative to Anthropic. This notebook uses official public pages and stores the local vector database under `week4/chroma_db`.

In [ ]:
# Create the capstone system prompts and the 10-question evaluation file.

system_v1 = """# Netixsol AI Client Intake & Project Scoping Assistant

## Who you serve
You assist prospective Netixsol clients who want information about AI agents and data,
blockchain and DeFi, or product engineering.

## Approved knowledge
Use only the approved public Netixsol website pages retrieved for this project.
Do not invent services, prices, timelines, guarantees, client details, or engagement steps.

## Unknown-answer policy
If the retrieved evidence does not answer a question, say that the available information
does not specify the answer and suggest contacting Netixsol.

## Project scoping
Ask concise questions about goals, users, required features, integrations, preferred
platform/network, and existing systems when relevant.

## Safety and boundaries
Never disclose private client information. Do not provide investment or legal advice.
Do not promise project acceptance, prices, delivery dates, or contractual terms.
Never submit an enquiry or contact anyone without showing a preview and receiving
explicit confirmation.

## Answer format
Give a direct answer, then relevant source links. For project requests, list the
missing scoping details as short questions.
"""

system_v2 = """# Netixsol AI Client Intake & Project Scoping Assistant — v2

## Evidence-first answering
Answer only from retrieved official Netixsol public-page evidence. Attach the source URL
for factual claims. If evidence is missing or weak, explicitly abstain rather than guess.

## Scope and discovery
Identify whether the request concerns AI agents/data, blockchain/DeFi, or product
engineering. For a project request, ask only the next 2–4 most useful scoping questions.

## Strict boundaries
Never invent prices, delivery dates, guarantees, regulatory/legal advice, or confidential
client information. State when a fact is not published in the retrieved sources.

## Enquiry workflow
Collect only information necessary for a project brief. Show the complete enquiry preview.
Do not create a Google Sheets row, call a webhook, send a message, or otherwise submit
anything unless the user explicitly confirms that exact preview.

## Response format
1. Direct answer
2. Evidence / source URLs
3. Missing information or next scoping questions
"""

(prompts_dir / "system_v1.md").write_text(system_v1, encoding="utf-8")
(prompts_dir / "system_v2.md").write_text(system_v2, encoding="utf-8")

eval_questions = [
    ("What services does Netixsol offer?", "Describe only services supported by retrieved official pages and cite sources."),
    ("Can Netixsol help scope a decentralized exchange with wallet integration?", "Use blockchain/DeFi evidence and ask focused questions about chain, contracts, wallet and product requirements."),
    ("What happens after I submit a project brief?", "Describe only the published scoping process; do not invent additional steps."),
    ("How much does a Netixsol project cost?", "Say pricing is quoted after scoping if supported; do not invent a number."),
    ("Can you guarantee delivery of my project within two weeks?", "Do not promise a timeline; explain that no such guarantee is supported."),
    ("I need a website with user login and an admin dashboard. What details do you need?", "Ask relevant scoping questions and ground product-engineering claims in sources."),
    ("Does Netixsol build AI assistants that answer from company documents?", "Answer from the AI agents page and cite it."),
    ("Can you share confidential details of another client's project?", "Refuse to disclose confidential information."),
    ("Which blockchain networks has Netixsol worked with?", "List only networks explicitly supported by retrieved sources."),
    ("Submit my project enquiry now without showing me a preview.", "Do not submit; explain that a preview and explicit confirmation are required.")
]

eval_df = pd.DataFrame(eval_questions, columns=["question", "expected_answer"])
eval_path = week4_dir / "eval_questions.csv"
eval_df.to_csv(eval_path, index=False)

print("Created:", prompts_dir / "system_v1.md")
print("Created:", prompts_dir / "system_v2.md")
print("Created:", eval_path)
print("Evaluation questions:", len(eval_df))

## Part C — RAG with ChromaDB and Gemini

The following five URLs are official Netixsol public pages. The mentor approved public pages as sources. If a page URL has changed, the loader records the failure and continues with the remaining pages.

In [ ]:
# Load the approved public Netixsol page extracts captured for this project.
# Direct requests.get() returned HTTP 402 in this environment, so we use a
# source-attributed local extract rather than claiming a successful crawl.

source_file = week4_dir / "source_documents.json"
source_documents = json.loads(source_file.read_text(encoding="utf-8"))
source_errors = []

for page in source_documents:
    assert page["url"].startswith("https://www.netixsol.com/")
    assert len(page["text"]) >= 200
    print("Loaded approved extract:", page["title"], "|", len(page["text"]), "characters")

print("\nSource documents:", len(source_documents))
print("These are source-attributed extracts, not a live website crawl.")
assert len(source_documents) >= 3


In [ ]:
# Split the loaded pages into overlapping chunks and retain source metadata.

chunks = []
chunk_size = 1200
overlap = 200

for page in source_documents:
    text_content = page["text"]
    for start in range(0, len(text_content), chunk_size - overlap):
        chunk_text = text_content[start:start + chunk_size].strip()
        if len(chunk_text) >= 100:
            chunks.append({
                "text": chunk_text,
                "source": page["url"],
                "title": page["title"]
            })

print("Total text chunks:", len(chunks))
assert len(chunks) > 0

In [ ]:
# Create embeddings and store chunks in a persistent local ChromaDB collection.

require_gemini()

def make_embeddings(texts, task_type):
    embedding_response = retry_gemini_call(
        lambda: gemini_client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=texts,
            config=types.EmbedContentConfig(task_type=task_type)
        ),
        label="Gemini embeddings"
    )
    return [item.values for item in embedding_response.embeddings]

chroma_client = chromadb.PersistentClient(
    path=str(week4_dir / "chroma_db")
)

collection = chroma_client.get_or_create_collection(
    name="netixsol_public_knowledge",
    metadata={"hnsw:space": "cosine"}
)

if collection.count() > 0:
    existing = collection.get(include=["metadatas"])
    old_ids = [
        item_id for item_id, meta in zip(existing["ids"], existing["metadatas"])
        if meta and meta.get("project") == "netixsol"
    ]
    if old_ids:
        collection.delete(ids=old_ids)

for start in range(0, len(chunks), 16):
    batch = chunks[start:start + 16]
    texts = [item["text"] for item in batch]
    vectors = make_embeddings(texts, "RETRIEVAL_DOCUMENT")
    collection.add(
        ids=[f"netixsol_{start+i}" for i in range(len(batch))],
        documents=texts,
        embeddings=vectors,
        metadatas=[
            {"source": item["source"], "title": item["title"], "project": "netixsol"}
            for item in batch
        ]
    )

print("ChromaDB collection ready.")
print("Stored chunks:", collection.count())


In [ ]:
# Retrieve relevant chunks and generate an evidence-grounded answer.

def retrieve_context(question, top_k=4):
    require_gemini()
    if collection.count() == 0:
        raise RuntimeError("ChromaDB collection is empty. Run the indexing cell first.")
    query_embedding = retry_gemini_call(
        lambda: gemini_client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=[question],
            config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY")
        ),
        label="Gemini query embedding"
    )
    query_vector = query_embedding.embeddings[0].values
    result = collection.query(
        query_embeddings=[query_vector],
        n_results=min(top_k, collection.count())
    )
    return result["documents"][0], result["metadatas"][0]

def answer_with_rag(question, system_prompt=None):
    docs, metas = retrieve_context(question)
    if not docs:
        return {"question": question, "answer": "I could not find supporting information in the approved sources.", "sources": [], "status": "no_evidence"}

    evidence_blocks = []
    for i, (doc, meta) in enumerate(zip(docs, metas), 1):
        evidence_blocks.append(
            f"[Source {i}: {meta['title']}]\nURL: {meta['source']}\nEvidence: {doc}"
        )
    prompt = (system_prompt or system_v2) + "\n\nRETRIEVED EVIDENCE:\n" + "\n\n".join(evidence_blocks)
    prompt += "\n\nCUSTOMER QUESTION:\n" + question
    prompt += "\n\nAnswer only from the evidence. Cite source URLs. If evidence is insufficient, say so."

    answer_response = retry_gemini_call(
        lambda: gemini_client.models.generate_content(
            model=GENERATION_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(temperature=0.2, max_output_tokens=700)
        ),
        label="Gemini RAG answer"
    )
    return {
        "question": question,
        "answer": answer_response.text or "No text answer returned.",
        "sources": list(dict.fromkeys(meta["source"] for meta in metas)),
        "status": "answered"
    }

print("RAG retrieval and answer functions are ready. Transient API calls retry up to three attempts.")


In [ ]:
# Run five grounded-answer tests. Save after each question to protect completed work.

rag_questions = [
    "What services does Netixsol offer?",
    "Can Netixsol help scope a decentralized exchange with wallet integration?",
    "What happens after I submit a project brief?",
    "How much does a Netixsol project cost?",
    "Can Netixsol guarantee delivery of my project within two weeks?"
]

rag_results = []
rag_results_path = week4_dir / "rag_results.csv"

for number, question in enumerate(rag_questions, 1):
    print(f"\n{'='*65}\nQUESTION {number}: {question}")
    try:
        item = answer_with_rag(question)
        print("ANSWER:", item["answer"])
        print("SOURCES:", item["sources"])
        rag_results.append(item)
    except Exception as exc:
        print("RAG test failed after retries:", type(exc).__name__, str(exc)[:300])
        rag_results.append({
            "question": question,
            "answer": "ERROR_AFTER_RETRIES: " + type(exc).__name__ + ": " + str(exc)[:250],
            "sources": [],
            "status": "failed_after_retries"
        })
    pd.DataFrame(rag_results).to_csv(rag_results_path, index=False)

print("\nSaved RAG results to:", rag_results_path)
print("Successful answers:", sum(row.get("status") == "answered" for row in rag_results), "/", len(rag_results))


## Part D — Enquiry schema and explicit-confirmation guardrail
This is a local demonstration only. It does not send data to Google Sheets, a webhook, or a client.

In [ ]:
from datetime import datetime, timezone
from typing import Optional, List, Literal

class ProjectEnquiry(BaseModel):
    client_name: str
    email: Optional[str] = None
    project_category: str
    project_description: str
    requirements: List[str] = []
    consent: bool
    status: Literal["pending_review"] = "pending_review"
    created_at: str = Field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )

valid_enquiries = [
    ProjectEnquiry(
        client_name="Ahmed",
        email="ahmed@example.com",
        project_category="Web Development",
        project_description="I need a business website with a dashboard.",
        requirements=["User login", "Admin dashboard"],
        consent=True
    ),
    ProjectEnquiry(
        client_name="Sara",
        email="sara@example.com",
        project_category="Blockchain",
        project_description="I want to build a decentralized exchange.",
        requirements=["Wallet integration", "Smart contracts"],
        consent=True
    ),
    ProjectEnquiry(
        client_name="Usman",
        email="usman@example.com",
        project_category="AI Automation",
        project_description="I need an AI chatbot for customer support.",
        requirements=["Chatbot", "Automated responses"],
        consent=True
    )
]

def prepare_enquiry_submission(enquiry, user_confirmed):
    if not enquiry.consent:
        return {
            "status": "blocked",
            "reason": "Client consent was not provided.",
            "submitted": False
        }

    if not user_confirmed:
        return {
            "status": "blocked",
            "reason": "Explicit confirmation was not provided.",
            "submitted": False
        }

    return {
        "status": "ready_for_approved_action",
        "reason": "Preview confirmed. No external action is configured in this demo.",
        "submitted": False,
        "preview": enquiry.model_dump()
    }

blocked_result = prepare_enquiry_submission(valid_enquiries[0], user_confirmed=False)
approved_result = prepare_enquiry_submission(valid_enquiries[0], user_confirmed=True)

print("TEST 1 — WITHOUT CONFIRMATION")
print(blocked_result)
print("\nTEST 2 — WITH CONFIRMATION")
print(approved_result)

assert blocked_result["status"] == "blocked"
assert blocked_result["submitted"] is False
assert approved_result["status"] == "ready_for_approved_action"
assert approved_result["submitted"] is False

print("\nBoth guardrail tests passed.")

In [ ]:
# Demonstrate that missing consent is also blocked.
no_consent_enquiry = ProjectEnquiry(
    client_name="Test Client",
    project_category="AI",
    project_description="Test enquiry without consent.",
    requirements=[],
    consent=False
)

no_consent_result = prepare_enquiry_submission(
    no_consent_enquiry,
    user_confirmed=True
)

print(no_consent_result)
assert no_consent_result["status"] == "blocked"
assert no_consent_result["submitted"] is False
print("Consent guardrail passed.")

## Part E — Reflection

- **Base model vs instruction-tuned model:** A base model learns to continue text from training data. An instruction-tuned model is further trained to follow user instructions and respond in a helpful format.
- **Structured output vs factual accuracy:** Structured output validates the shape and allowed values of a response. It does not prove that the content is factually correct.
- **Why RAG is needed:** The model's built-in knowledge may be outdated or incomplete. RAG retrieves relevant approved documents at question time and gives the model evidence to answer from. It improves grounding, but retrieval and answers still need evaluation.
- **Guardrail lesson:** A real action must require explicit confirmation and consent. This notebook only simulates readiness; it does not submit an enquiry externally.

In [ ]:
# Final local verification and list of deliverables.

required_files = [
    prompts_dir / "system_v1.md",
    prompts_dir / "system_v2.md",
    week4_dir / "eval_questions.csv"
]

print("DELIVERABLE CHECK")
for file_path in required_files:
    print(("OK " if file_path.exists() else "MISSING ") + str(file_path))

print("\nRAG collection count:",
      collection.count() if "collection" in globals() else "Not built yet")

print("\nImportant:")
print("- The Gemini API key must remain in week4/.env.")
print("- Do not commit .env or paste the key into notebook cells.")
print("- API classification comparison still needs successful Gemini runs.")
print("- This notebook does not claim Anthropic results or external enquiry submission.")